# 105 Python intermediate
## Własne moduły i biblioteki

_Kamil Bartocha_ - wersja 2.0

## Rozkład jazdy

1. 🔹 **Moduły i pakiety** - import, `__init__.py`, struktura
2. 🔹 **Tworzenie biblioteki** - projekt `datareader` krok po kroku
3. 🔹 **Testy i instalacja** - `unittest`, `pip install .`, użycie

🐍 Każda sekcja zawiera ćwiczenia.

---

## 1. 🔹 Moduły i pakiety

**Moduł (module)** to pojedynczy plik `.py` zawierający funkcje,
klasy i zmienne. **Pakiet (package)** to katalog z modułami
i plikiem `__init__.py`.

| | Moduł | Pakiet |
|---|---|---|
| Struktura | plik `utils.py` | katalog `utils/` |
| Wymagany plik | brak | `__init__.py` |
| Import | `import utils` | `import utils` lub `from utils import X` |
| Zastosowanie | mała biblioteka | duży projekt, wiele modułów |

**Style importu:**
```python
import math                    # cały moduł
from math import sqrt          # konkretna funkcja
from math import sqrt as sq    # z aliasem
from datareader import ReadXML # z własnego pakietu
```

> 💡 Plik `__init__.py` może być pusty - jego sama obecność sprawia,
> że Python traktuje katalog jako pakiet. Możemy w nim też kontrolować,
> co jest widoczne po `from pakiet import *`.

In [ ]:
# Moduł to zwykły plik .py - możemy go zaimportować
import math
import os
import sys

print(math.sqrt(16))         # 4.0
print(os.getcwd())           # bieżący katalog
print(sys.version[:5])       # wersja Pythona

# Importowanie konkretnej funkcji
from math import pi, factorial
print(pi)                    # 3.141592653589793
print(factorial(5))          # 120

### Rola pliku `__init__.py`

Plik `__init__.py` pełni trzy funkcje:

1. **Oznacza katalog jako pakiet** - bez niego Python nie traktuje
   katalogu jako modułu do importowania
2. **Kontroluje publiczny interfejs** - definiujemy, co importujemy
   przy `from pakiet import *`
3. **Upraszcza import** - możemy wyeksponować klasy z zagnieżdżonych
   modułów na poziomie pakietu

```python
# datareader/__init__.py
from .xml_reader import ReadXML    # . oznacza bieżący pakiet
from .json_reader import ReadJSON
```

Dzięki temu użytkownik importuje prosto:
```python
from datareader import ReadXML, ReadJSON  # zamiast:
from datareader.xml_reader import ReadXML  # (dłużej)
```

---

### 🐍 Ćwiczenia - Moduły i pakiety

**Ćw. 1.** Napisz poniżej funkcje modułu `mathutils`:
- `add(a, b)` - suma
- `subtract(a, b)` - różnica
- `multiply(a, b)` - iloczyn
- `average(*args)` - średnia

Normalnie byłyby w osobnym pliku `mathutils.py`. Tutaj piszemy
je inline, żeby przetestować.

**Ćw. 2.** Napisz funkcje modułu `stringutils`:
- `clean(text)` - usuwa zbędne spacje (`.strip()`)
- `capitalize_words(text)` - każde słowo z wielką literą
- `count_words(text)` - liczba słów w tekście

**Ćw. 3. *(Trudniejsze)*** Napisz zawartość pliku `__init__.py`
dla pakietu `myutils`, który eksponuje wszystkie funkcje z obu
modułów `mathutils` i `stringutils`.
Następnie pokaż, jak użytkownik zaimportuje `average` i `clean`
używając tylko `from myutils import ...`.

In [ ]:
# Ćw. 1: Funkcje mathutils
def add(a, b):
    ...

def subtract(a, b):
    ...

def multiply(a, b):
    ...

def average(*args):
    ...


print(add(3, 4))           # 7
print(subtract(10, 3))     # 7
print(multiply(3, 4))      # 12
print(average(1, 2, 3, 4)) # 2.5

In [ ]:
# Ćw. 2: Funkcje stringutils
def clean(text):
    ...

def capitalize_words(text):
    ...

def count_words(text):
    ...


print(clean("  hello world  "))        # "hello world"
print(capitalize_words("hello world")) # "Hello World"
print(count_words("one two three"))    # 3

In [ ]:
# Ćw. 3: Zawartość __init__.py dla pakietu myutils
# Gdyby struktura była:
#   myutils/
#       __init__.py
#       mathutils.py
#       stringutils.py
#
# Zawartość __init__.py:
# from .mathutils import ...
# from .stringutils import ...

# Wpisz zawartość __init__.py jako komentarze:
# ...

# Jak użytkownik importuje:
# from myutils import ...

---

## 2. 🔹 Tworzenie biblioteki - projekt `datareader`

Tworzymy bibliotekę `datareader` do odczytu plików XML i JSON.
Pozwoli to używać tego samego kodu w wielu projektach.

### Dlaczego tworzymy własną bibliotekę?

- **Ponowne użycie kodu** - ten sam kod działa w wielu projektach
- **Organizacja** - logika podzielona na moduły, łatwa w utrzymaniu
- **Testowanie** - izolowany kod łatwiej testować automatycznie
- **Udostępnianie** - możemy opublikować na PyPI (`pip install`)

### Struktura projektu

```
datareader/               <- główny katalog projektu
├── datareader/           <- pakiet (kod biblioteki)
│   ├── __init__.py
│   ├── xml_reader.py
│   └── json_reader.py
├── tests/
│   ├── example1.xml
│   ├── example1.json
│   ├── test_xml_reader.py
│   └── test_json_reader.py
├── README.md
├── setup.py
└── LICENSE
```

### Krok 1: Implementacja klasy `ReadXML`

`datareader/xml_reader.py`

In [ ]:
import xml.etree.ElementTree as ET


class ReadXML:
    def __init__(self, file_path):
        self.file_path = file_path
        self.tree = None
        self.root = None

    def load_xml(self):
        try:
            self.tree = ET.parse(self.file_path)
            self.root = self.tree.getroot()
            print("XML file loaded successfully.")
        except ET.ParseError as e:
            print(f"Error parsing XML file: {e}")
        except FileNotFoundError:
            print(f"File not found: {self.file_path}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

    def _get_element_by_path(self, path):
        parts = path.split(".")
        current_elem = self.root
        for part in parts:
            if part.startswith("root"):
                continue
            current_elem = current_elem.find(part)
            if current_elem is None:
                return None
        return current_elem

    def get_value(self, path):
        elem = self._get_element_by_path(path)
        if elem is not None:
            return elem.text
        else:
            return "Key not found"

### Krok 2: Implementacja klasy `ReadJSON`

`datareader/json_reader.py`

In [ ]:
import json


class ReadJSON:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def load_json(self):
        try:
            with open(self.file_path, 'r') as file:
                self.data = json.load(file)
            print("JSON file loaded successfully.")
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON file: {e}")
        except FileNotFoundError:
            print(f"File not found: {self.file_path}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

    def _get_element_by_path(self, path):
        parts = path.split(".")
        current_elem = self.data
        for part in parts:
            if '[' in part and ']' in part:
                key   = part[:part.index('[')]
                index = int(part[part.index('[') + 1:part.index(']')])
                current_elem = current_elem.get(key, None)
                if isinstance(current_elem, list):
                    current_elem = current_elem[index]
                else:
                    return None
            else:
                current_elem = current_elem.get(part, None)
            if current_elem is None:
                return None
        return current_elem

    def get_value(self, path):
        elem = self._get_element_by_path(path)
        if elem is not None:
            return elem
        else:
            return "Key not found"

### Krok 3: Plik `__init__.py`

`datareader/__init__.py` - eksponuje klasy na poziomie pakietu:

In [ ]:
# datareader/__init__.py
from .xml_reader import ReadXML
from .json_reader import ReadJSON

# Dzięki temu użytkownik pisze:
# from datareader import ReadXML, ReadJSON
# zamiast:
# from datareader.xml_reader import ReadXML

### Krok 4: Plik `setup.py`

Plik `setup.py` umożliwia instalację biblioteki przez `pip`.
Po jego stworzeniu uruchamiamy w katalogu projektu:

```bash
pip install .          # instalacja lokalna
pip install -e .       # instalacja edytowalna (zmiany natychmiast widoczne)
```

In [ ]:
# setup.py
from setuptools import setup, find_packages

setup(
    name="datareader",
    version="0.1.0",
    description="A library for reading and processing XML and JSON data.",
    long_description=open("README.md").read(),
    long_description_content_type="text/markdown",
    author="Kamil Bartocha",
    packages=find_packages(),
    classifiers=[
        "Programming Language :: Python :: 3",
        "License :: OSI Approved :: MIT License",
        "Operating System :: OS Independent",
    ],
    python_requires='>=3.6',
)

> 💡 Nowoczesne projekty zastępują `setup.py` plikiem
> `pyproject.toml` (PEP 517/518). Oba podejścia działają,
> ale `pyproject.toml` jest standardem od Pythona 3.11+.

---

### 🐍 Ćwiczenia - Struktura biblioteki

**Ćw. 4.** Poniżej masz strukturę pakietu `calculator`. Uzupełnij
zawartość `__init__.py` tak, żeby użytkownik mógł pisać:
`from calculator import add, subtract, multiply`

```
calculator/
├── __init__.py      <- uzupełnij
├── basic_ops.py     <- zawiera: add, subtract, multiply
└── advanced_ops.py  <- zawiera: power, sqrt
```

**Ćw. 5.** Napisz uproszczoną wersję `ReadJSON` - klasę
`SimpleJSON`, która:
- w `__init__` przyjmuje `file_path`
- ma metodę `load()` wczytującą plik do `self.data`
- ma metodę `get(key)` zwracającą wartość dla klucza
  (tylko jeden poziom - nie zagnieżdżone ścieżki)

**Ćw. 6. *(Trudniejsze)*** Rozszerz `SimpleJSON` o metodę
`keys()` zwracającą listę kluczy najwyższego poziomu i metodę
`summary()` wypisującą klucze i typy ich wartości.

In [ ]:
# Ćw. 4: Zawartość __init__.py
# Wpisz jako komentarze:
#
# calculator/__init__.py
# from .basic_ops import ...
# from .advanced_ops import ...
#
# ...

In [ ]:
# Ćw. 5: SimpleJSON
import json


class SimpleJSON:
    def __init__(self, file_path):
        ...

    def load(self):
        ...

    def get(self, key):
        ...


# Test z danymi inline (bez pliku)
import io

class SimpleJSONFromString(SimpleJSON):
    """Wersja testowa - wczytuje z napisu zamiast pliku."""
    def __init__(self, json_string):
        self.data = json.loads(json_string)
        self.file_path = None

    def load(self):
        pass  # dane już załadowane w __init__


sample = '{"name": "Alice", "age": 30, "city": "Warsaw"}'
reader = SimpleJSONFromString(sample)
print(reader.get("name"))   # Alice
print(reader.get("age"))    # 30

In [ ]:
# Ćw. 6: Rozszerzenie SimpleJSON o keys() i summary()
class SimpleJSONExtended(SimpleJSONFromString):
    def keys(self):
        ...

    def summary(self):
        # hint: type(val).__name__ zwraca nazwę typu jako string
        ...


sample = '{"name": "Alice", "age": 30, "scores": [95, 87]}'
reader = SimpleJSONExtended(sample)
print(reader.keys())    # ['name', 'age', 'scores']
reader.summary()
# name: str
# age: int
# scores: list

---

## 3. 🔹 Testy i instalacja

Dobra biblioteka zawiera testy jednostkowe (unit tests), które
automatycznie weryfikują poprawność każdej funkcji i klasy.

Python ma wbudowany moduł `unittest`. Testy umieszczamy w katalogu
`tests/` w plikach zaczynających się od `test_`.

| Element | Opis |
|---|---|
| `unittest.TestCase` | klasa bazowa dla przypadków testowych |
| `def test_*(self)` | metoda testowa - musi zaczynać się od `test_` |
| `self.assertEqual(a, b)` | sprawdza czy `a == b` |
| `self.assertIsNotNone(x)` | sprawdza czy `x is not None` |
| `self.assertRaises(Exc, fn)` | sprawdza czy funkcja rzuca wyjątek |

Uruchamiamy testy:
```bash
python -m pytest tests/       # z pytest
python -m unittest discover   # z unittest
```

### Testy dla `datareader`

`tests/test_xml_reader.py`

In [ ]:
import unittest
from datareader.xml_reader import ReadXML


class TestReadXML(unittest.TestCase):

    def test_load_xml(self):
        reader = ReadXML('tests/example1.xml')
        reader.load_xml()
        self.assertIsNotNone(reader.root)

    def test_get_value(self):
        reader = ReadXML('tests/example1.xml')
        reader.load_xml()
        value = reader.get_value('root.item.name')
        self.assertEqual(value, 'Item 1')

    def test_file_not_found(self):
        reader = ReadXML('nonexistent.xml')
        reader.load_xml()   # nie powinno rzucić wyjątku - obsługujemy go
        self.assertIsNone(reader.root)


if __name__ == "__main__":
    unittest.main()

`tests/test_json_reader.py`

In [ ]:
import unittest
from datareader.json_reader import ReadJSON


class TestReadJSON(unittest.TestCase):

    def test_load_json(self):
        reader = ReadJSON('tests/example1.json')
        reader.load_json()
        self.assertIsNotNone(reader.data)

    def test_get_value(self):
        reader = ReadJSON('tests/example1.json')
        reader.load_json()
        value = reader.get_value('root.item[0].name')
        self.assertEqual(value, 'Item 1')

    def test_key_not_found(self):
        reader = ReadJSON('tests/example1.json')
        reader.load_json()
        value = reader.get_value('nonexistent.key')
        self.assertEqual(value, 'Key not found')


if __name__ == "__main__":
    unittest.main()

### Instalacja i użycie

Po stworzeniu `setup.py` instalujemy bibliotekę lokalnie:

```bash
cd datareader/           # katalog z setup.py
pip install -e .         # -e = editable, zmiany od razu widoczne
```

Teraz możemy importować `datareader` w dowolnym projekcie:

In [ ]:
# Po instalacji: from datareader import ReadXML, ReadJSON
# Tutaj używamy lokalnej ścieżki do demonstracji

import sys
sys.path.insert(0, '105_library/datareader')

from datareader import ReadXML, ReadJSON

# Odczyt XML
xml_reader = ReadXML('105_library/datareader/tests/example1.xml')
xml_reader.load_xml()
print(xml_reader.get_value('root.item.name'))   # Item 1

# Odczyt JSON
json_reader = ReadJSON('105_library/datareader/tests/example1.json')
json_reader.load_json()
print(json_reader.get_value('root.item[0].name'))   # Item 1
print(json_reader.get_value('root.item[1].price'))  # 20.00

---

### 🐍 Ćwiczenia - Testy i instalacja

**Ćw. 7.** Napisz testy `unittest` dla funkcji z `mathutils`
(ćw. 1). Przynajmniej jeden test na funkcję:
- `test_add`, `test_subtract`, `test_multiply`, `test_average`

**Ćw. 8.** Napisz test, który sprawdza obsługę błędu:
wywołaj `ReadJSON` z nieistniejącą ścieżką i sprawdź,
że `reader.data` pozostaje `None` (błąd obsłużony wewnętrznie).

**Ćw. 9. *(Trudniejsze)*** Napisz zawartość `setup.py` dla
pakietu `myutils` (ćw. 3). Uwzględnij:
- `name`, `version`, `description`
- `packages=find_packages()`
- `python_requires='>=3.8'`

Następnie wpisz polecenia instalacji lokalnej (jako komentarze).

In [ ]:
# Ćw. 7: Testy dla mathutils
import unittest


# Wklejamy funkcje z ćw. 1 (normalnie byłyby w osobnym pliku)
def add(a, b): return a + b
def subtract(a, b): return a - b
def multiply(a, b): return a * b
def average(*args): return sum(args) / len(args)


class TestMathUtils(unittest.TestCase):
    def test_add(self):
        ...

    def test_subtract(self):
        ...

    def test_multiply(self):
        ...

    def test_average(self):
        ...


# Uruchom testy:
unittest.main(argv=[''], exit=False, verbosity=2)

In [ ]:
# Ćw. 8: Test obsługi błędu FileNotFoundError
import unittest


class TestReadJSONErrors(unittest.TestCase):
    def test_file_not_found(self):
        # hint: stwórz ReadJSON z nieistniejącą ścieżką,
        # wywołaj load_json(), sprawdź self.data is None
        ...


unittest.main(argv=[''], exit=False, verbosity=2)

In [ ]:
# Ćw. 9: setup.py dla myutils
# from setuptools import setup, find_packages
#
# setup(
#     name=...
#     version=...
#     description=...
#     packages=find_packages(),
#     python_requires='>=3.8',
# )

# Polecenia instalacji lokalnej:
# ...